# Drone Gate Detection - CNN Model
## Competition: ae4353-y25

**Hardware Optimization:**
- T4 GPU: 15GB VRAM
- CPU: 30GB RAM

This notebook implements an improved CNN architecture for detecting gates in drone racing images.

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q h5py hdf5plugin torch torchvision tqdm pandas numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import sigmoid_focal_loss, complete_box_iou_loss
import torchvision

import h5py
import hdf5plugin
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import os
import gc

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Enable cudnn benchmarking for faster training
torch.backends.cudnn.benchmark = True

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration

In [ ]:
# Paths (Kaggle environment)
DATA_ROOT = '/kaggle/input/ae4353-y25'  # Update this to your Kaggle dataset path

# IMPORTANT: Update these paths to match your actual training file paths
TRAIN_FILE_PATHS = [
    os.path.join(DATA_ROOT, 'train_file_1.h5'),
    os.path.join(DATA_ROOT, 'train_file_2.h5'),
    os.path.join(DATA_ROOT, 'train_file_3.h5'),
    os.path.join(DATA_ROOT, 'train_file_4.h5'),
    os.path.join(DATA_ROOT, 'train_file_5.h5'),
    os.path.join(DATA_ROOT, 'train_file_6.h5'),
    os.path.join(DATA_ROOT, 'train_file_7.h5'),
    os.path.join(DATA_ROOT, 'train_file_8.h5'),
    os.path.join(DATA_ROOT, 'train_file_9.h5'),
    os.path.join(DATA_ROOT, 'train_file_10.h5'),
]

TEST_SET_PATH = os.path.join(DATA_ROOT, 'test_set.h5')

# Model hyperparameters - Optimized for T4 GPU (15GB VRAM)
CONFIG = {
    'grid_size': 14,
    'num_anchors': 2,
    'batch_size': 16,  # Optimized for T4 15GB VRAM
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'num_epochs': 50,
    'num_workers': 4,  # Optimized for CPU with 30GB RAM
    'pin_memory': True,
    'early_stopping_patience': 10,
    'grad_clip': 1.0,
    # Loss weights
    'lambda_coord': 5.0,
    'lambda_noobj': 0.5,
    'lambda_obj': 1.0,
    'lambda_size': 2.0,
    # Inference
    'conf_thresh': 0.25,
    'nms_thresh': 0.4,
    'use_tta': False,  # Test-Time Augmentation (slower but more accurate)
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Configuration: {CONFIG}")
print(f"\nNumber of training files: {len(TRAIN_FILE_PATHS)}")

## 3. Dataset Classes

In [ ]:
class MultiFileGateDataset(Dataset):
    """Training dataset that loads from multiple H5 files"""
    def __init__(self, h5_file_paths, grid_size=14, num_anchors=2):
        self.grid_size = grid_size
        self.num_anchors = num_anchors
        
        # Load all images and labels from multiple files
        all_images = []
        all_labels = []
        
        print(f"Loading {len(h5_file_paths)} training files...")
        for i, h5_path in enumerate(h5_file_paths):
            print(f"  Loading file {i+1}/{len(h5_file_paths)}: {os.path.basename(h5_path)}")
            with h5py.File(h5_path, 'r') as f:
                images = f['images'][:]
                labels = f['labels'][:]
                all_images.append(images)
                all_labels.append(labels)
                print(f"    - Loaded {len(images)} images")
        
        # Concatenate all data
        self.images = np.concatenate(all_images, axis=0)
        self.labels = np.concatenate(all_labels, axis=0)
        
        print(f"\nTotal training images: {len(self.images)}")
        print(f"Image shape: {self.images.shape}")
        print(f"Label shape: {self.labels.shape}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Get image and normalize
        image = self.images[idx].astype(np.float32) / 255.0
        
        # Convert HWC to CHW format
        image = np.transpose(image, (2, 0, 1))
        
        # Normalize with ImageNet stats
        mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
        image = (image - mean) / std
        
        # Get label
        label = self.labels[idx]
        
        return torch.from_numpy(image).float(), torch.from_numpy(label).float()


class TestDataset(Dataset):
    """Test dataset for inference"""
    def __init__(self, h5_path):
        # Load test images into memory
        with h5py.File(h5_path, 'r') as f:
            self.images = f['images'][:]
        
        print(f"Loaded {len(self.images)} test images")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Get image and normalize
        image = self.images[idx].astype(np.float32) / 255.0
        
        # Convert HWC to CHW format
        image = np.transpose(image, (2, 0, 1))
        
        # Normalize with ImageNet stats
        mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
        image = (image - mean) / std
        
        return torch.from_numpy(image).float()

## 4. Model Architecture

In [ ]:
class ImprovedGateCNN(nn.Module):
    """Enhanced CNN architecture with attention and residual connections"""
    def __init__(self, grid_size=14, num_anchors=2):
        super(ImprovedGateCNN, self).__init__()
        self.S = grid_size
        self.num_anchors = num_anchors
        
        # Encoder with residual blocks
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        self.conv5 = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        
        # Spatial Attention Module
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(512, 256, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 1, 1),
            nn.Sigmoid()
        )
        
        # Calculate feature map size after pooling
        # Input: 480x640 -> after 5 pooling: 15x20
        self.feature_size = 512 * 15 * 20
        
        # FC layers with residual connection
        self.fc1 = nn.Linear(self.feature_size, 4096)
        self.fc2 = nn.Linear(4096, 2048)
        self.fc3 = nn.Linear(2048, self.S * self.S * num_anchors * 5)
        
        self.bn_fc1 = nn.BatchNorm1d(4096)
        self.bn_fc2 = nn.BatchNorm1d(2048)
        
    def forward(self, x):
        # Encoder with progressive downsampling
        x = self.pool(self.conv1(x))  # 240x320
        x = self.pool(self.conv2(x))  # 120x160
        x = self.pool(self.conv3(x))  # 60x80
        x = self.pool(self.conv4(x))  # 30x40
        x = self.pool(self.conv5(x))  # 15x20
        
        # Apply spatial attention
        attention = self.spatial_attention(x)
        x = x * attention
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers with dropout and batch norm
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn_fc2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)
        
        # Reshape to grid format: [batch, grid_y, grid_x, num_anchors, 5]
        return x.reshape(-1, self.S, self.S, self.num_anchors, 5)

## 5. Loss Function

In [ ]:
class EnhancedLoss(nn.Module):
    """Improved loss function with focal loss and CIoU"""
    def __init__(self, grid_size=14, lambda_coord=5.0, lambda_noobj=0.5, 
                 lambda_obj=1.0, lambda_size=2.0):
        super(EnhancedLoss, self).__init__()
        self.S = grid_size
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.lambda_obj = lambda_obj
        self.lambda_size = lambda_size

    def forward(self, predictions, target):
        """
        predictions: [batch, S, S, num_anchors, 5]
        target: [batch, S, S, num_anchors, 5]
        """
        # Object mask: where target confidence = 1
        obj_mask = target[..., 0] == 1
        noobj_mask = target[..., 0] == 0
        
        # 1. Objectness loss (Focal Loss for class imbalance)
        if obj_mask.sum() > 0:
            obj_preds = predictions[obj_mask][..., 0]
            obj_targets = target[obj_mask][..., 0]
            obj_loss = sigmoid_focal_loss(
                obj_preds, 
                obj_targets, 
                alpha=0.25,
                gamma=2.0,
                reduction='mean'
            )
        else:
            obj_loss = torch.tensor(0.0, device=predictions.device)
        
        # 2. No-object loss
        if noobj_mask.sum() > 0:
            noobj_preds = predictions[noobj_mask][..., 0]
            noobj_targets = target[noobj_mask][..., 0]
            noobj_loss = sigmoid_focal_loss(
                noobj_preds,
                noobj_targets,
                alpha=0.75,
                gamma=2.0,
                reduction='mean'
            )
        else:
            noobj_loss = torch.tensor(0.0, device=predictions.device)
        
        # 3. Coordinate loss (only for boxes that exist)
        if obj_mask.sum() > 0:
            box_preds = predictions[obj_mask][..., 1:5]
            box_targets = target[obj_mask][..., 1:5]
            
            # Ensure positive width/height
            w_pred = torch.abs(box_preds[..., 2]) + 1e-6
            h_pred = torch.abs(box_preds[..., 3]) + 1e-6
            x_pred = box_preds[..., 0]
            y_pred = box_preds[..., 1]
            
            w_targ = box_targets[..., 2] + 1e-6
            h_targ = box_targets[..., 3] + 1e-6
            x_targ = box_targets[..., 0]
            y_targ = box_targets[..., 1]
            
            # Convert to x1y1x2y2 format for CIoU
            pred_x1 = x_pred - w_pred / 2
            pred_y1 = y_pred - h_pred / 2
            pred_x2 = x_pred + w_pred / 2
            pred_y2 = y_pred + h_pred / 2
            
            targ_x1 = x_targ - w_targ / 2
            targ_y1 = y_targ - h_targ / 2
            targ_x2 = x_targ + w_targ / 2
            targ_y2 = y_targ + h_targ / 2
            
            pred_boxes = torch.stack([pred_x1, pred_y1, pred_x2, pred_y2], dim=-1)
            targ_boxes = torch.stack([targ_x1, targ_y1, targ_x2, targ_y2], dim=-1)
            
            # CIoU loss
            ciou_loss = complete_box_iou_loss(pred_boxes, targ_boxes, reduction='mean')
            
            # Additional size loss
            size_loss = F.mse_loss(torch.sqrt(w_pred), torch.sqrt(w_targ)) + \
                       F.mse_loss(torch.sqrt(h_pred), torch.sqrt(h_targ))
            
            # Center coordinate loss
            center_loss = F.mse_loss(x_pred, x_targ) + F.mse_loss(y_pred, y_targ)
        else:
            ciou_loss = torch.tensor(0.0, device=predictions.device)
            size_loss = torch.tensor(0.0, device=predictions.device)
            center_loss = torch.tensor(0.0, device=predictions.device)
        
        # Combine all losses
        total_loss = (
            self.lambda_obj * obj_loss +
            self.lambda_noobj * noobj_loss +
            self.lambda_coord * (ciou_loss + center_loss) +
            self.lambda_size * size_loss
        )
        
        return total_loss, {
            'obj_loss': obj_loss.item(),
            'noobj_loss': noobj_loss.item(),
            'ciou_loss': ciou_loss.item(),
            'size_loss': size_loss.item(),
            'total_loss': total_loss.item()
        }

## 6. Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, epoch):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    loss_components = {'obj_loss': 0, 'noobj_loss': 0, 'ciou_loss': 0, 'size_loss': 0}
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # Forward pass
        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()
        outputs = model(images)
        loss, loss_dict = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        
        optimizer.step()
        
        # Update metrics
        running_loss += loss.item()
        for key in loss_components:
            loss_components[key] += loss_dict[key]
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{running_loss / (batch_idx + 1):.4f}",
            'obj': f"{loss_components['obj_loss'] / (batch_idx + 1):.4f}"
        })
    
    # Calculate average losses
    avg_loss = running_loss / len(train_loader)
    for key in loss_components:
        loss_components[key] /= len(train_loader)
    
    return avg_loss, loss_components


def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    loss_components = {'obj_loss': 0, 'noobj_loss': 0, 'ciou_loss': 0, 'size_loss': 0}
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Validating'):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(images)
            loss, loss_dict = criterion(outputs, labels)
            
            running_loss += loss.item()
            for key in loss_components:
                loss_components[key] += loss_dict[key]
    
    avg_loss = running_loss / len(val_loader)
    for key in loss_components:
        loss_components[key] /= len(val_loader)
    
    return avg_loss, loss_components

## 7. Inference Functions

In [ ]:
def get_bboxes_from_grid_pred(grid_preds, S=14, conf_thresh=0.25, nms_thresh=0.4):
    """
    Extract bounding boxes from grid predictions with NMS
    """
    pred_bboxes = []
    
    # Handle multiple anchors
    if len(grid_preds.shape) == 4:
        num_anchors = grid_preds.shape[2]
    else:
        num_anchors = 1
        grid_preds = grid_preds.unsqueeze(2)
    
    for i in range(S):
        for j in range(S):
            for a in range(num_anchors):
                conf = torch.sigmoid(grid_preds[i, j, a, 0])
                
                if conf > conf_thresh:
                    xc, yc, w, h = grid_preds[i, j, a, 1:]
                    
                    # Convert to absolute coordinates
                    x = (j + torch.sigmoid(xc)) / S
                    y = (i + torch.sigmoid(yc)) / S
                    w = torch.abs(w)
                    h = torch.abs(h)
                    
                    # Convert to x1y1x2y2 format
                    x1 = x - w / 2
                    y1 = y - h / 2
                    x2 = x + w / 2
                    y2 = y + h / 2
                    
                    pred_bboxes.append([x1.item(), y1.item(), x2.item(), y2.item(), conf.item()])
    
    # Apply NMS if multiple boxes detected
    if len(pred_bboxes) > 1:
        pred_bboxes_tensor = torch.tensor(pred_bboxes, device=grid_preds.device)
        boxes = pred_bboxes_tensor[:, :4]
        scores = pred_bboxes_tensor[:, 4]
        
        # Apply NMS
        keep_indices = torchvision.ops.nms(boxes, scores, nms_thresh)
        pred_bboxes_tensor = pred_bboxes_tensor[keep_indices]
        
        # Convert back to xywh format
        final_boxes = []
        for box in pred_bboxes_tensor:
            x1, y1, x2, y2, conf = box
            final_boxes.append([x1.item(), y1.item(), (x2 - x1).item(), (y2 - y1).item()])
        
        return final_boxes
    elif len(pred_bboxes) == 1:
        box = pred_bboxes[0]
        x1, y1, x2, y2, conf = box
        return [[x1, y1, x2 - x1, y2 - y1]]
    
    return []


def predict_with_tta(model, image, device, S=14):
    """Test-Time Augmentation for more robust predictions"""
    predictions = []
    
    # Original
    with torch.no_grad():
        pred = model(image.to(device))
        predictions.append(pred)
    
    # Horizontal flip
    image_flipped = torch.flip(image, dims=[3])
    with torch.no_grad():
        pred_flipped = model(image_flipped.to(device))
        # Un-flip the prediction (flip x-coordinate)
        pred_flipped = torch.flip(pred_flipped, dims=[2])
        predictions.append(pred_flipped)
    
    # Average predictions
    avg_pred = torch.mean(torch.stack(predictions), dim=0)
    return avg_pred

## 8. Load Data

In [ ]:
# Create datasets from multiple training files
print("Loading training data from multiple files...")
train_dataset = MultiFileGateDataset(
    TRAIN_FILE_PATHS, 
    grid_size=CONFIG['grid_size'],
    num_anchors=CONFIG['num_anchors']
)

# Create data loaders with optimized settings
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory'],
    persistent_workers=True,  # Keep workers alive between epochs
    prefetch_factor=2  # Prefetch 2 batches per worker
)

print(f"\nTraining set: {len(train_dataset)} images")
print(f"Batches per epoch: {len(train_loader)}")

## 9. Initialize Model and Optimizer

In [ ]:
# Initialize model
model = ImprovedGateCNN(
    grid_size=CONFIG['grid_size'],
    num_anchors=CONFIG['num_anchors']
)

# Move model to device
model = model.to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Initialize loss function
criterion = EnhancedLoss(
    grid_size=CONFIG['grid_size'],
    lambda_coord=CONFIG['lambda_coord'],
    lambda_noobj=CONFIG['lambda_noobj'],
    lambda_obj=CONFIG['lambda_obj'],
    lambda_size=CONFIG['lambda_size']
)

# Initialize optimizer with AdamW (better than Adam)
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

# Learning rate scheduler with warmup
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,  # Initial restart period
    T_mult=2,  # Period multiplier
    eta_min=1e-6
)

print("Model, loss, and optimizer initialized successfully!")

## 10. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'learning_rate': []
}

best_loss = float('inf')
epochs_no_improve = 0

print("\n" + "="*60)
print("Starting Training")
print("="*60)

for epoch in range(1, CONFIG['num_epochs'] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['num_epochs']}")
    print("-" * 60)
    
    # Train
    train_loss, train_components = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['learning_rate'].append(current_lr)
    
    # Print epoch summary
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  - Obj Loss: {train_components['obj_loss']:.4f}")
    print(f"  - NoObj Loss: {train_components['noobj_loss']:.4f}")
    print(f"  - CIoU Loss: {train_components['ciou_loss']:.4f}")
    print(f"  - Size Loss: {train_components['size_loss']:.4f}")
    print(f"  Learning Rate: {current_lr:.6f}")
    
    # Save best model
    if train_loss < best_loss:
        best_loss = train_loss
        epochs_no_improve = 0
        
        # Save checkpoint
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': train_loss,
            'grid_size': CONFIG['grid_size'],
            'num_anchors': CONFIG['num_anchors'],
            'config': CONFIG
        }
        torch.save(checkpoint, 'best_model.pth')
        print(f"  ✓ Best model saved! (Loss: {best_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement for {epochs_no_improve} epoch(s)")
    
    # Early stopping
    if epochs_no_improve >= CONFIG['early_stopping_patience']:
        print(f"\nEarly stopping triggered after {epoch} epochs")
        break
    
    # Clear cache to prevent memory issues
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("\n" + "="*60)
print("Training Complete!")
print(f"Best Loss: {best_loss:.4f}")
print("="*60)

## 11. Load Test Data and Generate Predictions

In [ ]:
# Load best model
print("Loading best model for inference...")
checkpoint = torch.load('best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")

In [ ]:
# Load test dataset
print("\nLoading test data...")
test_dataset = TestDataset(TEST_SET_PATH)

# Smaller batch size for inference to save memory
test_loader = DataLoader(
    test_dataset,
    batch_size=32,  # Larger batch for faster inference
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory']
)

print(f"Test set: {len(test_dataset)} images")

In [ ]:
# Generate predictions
print("\nGenerating predictions...")
predictions = []

with torch.no_grad():
    for images in tqdm(test_loader, desc='Predicting'):
        images = images.to(device, non_blocking=True)
        
        # Use TTA if enabled
        if CONFIG['use_tta']:
            outputs = predict_with_tta(model, images, device, S=CONFIG['grid_size'])
        else:
            outputs = model(images)
        
        # Process each image in batch
        for i in range(outputs.shape[0]):
            bboxes = get_bboxes_from_grid_pred(
                outputs[i], 
                S=CONFIG['grid_size'], 
                conf_thresh=CONFIG['conf_thresh'],
                nms_thresh=CONFIG['nms_thresh']
            )
            predictions.append(bboxes)

print(f"Generated predictions for {len(predictions)} images")

## 12. Format and Save Submission

In [ ]:
# Format predictions into submission string format
prediction_strings = []

for bboxes in predictions:
    pred_str = ''
    for bbox in bboxes:
        x, y, w, h = bbox
        
        # Convert to corner format (x1,y1,x2,y2,x3,y3,x4,y4)
        x1, y1 = x, y
        x2, y2 = x + w, y
        x3, y3 = x + w, y + h
        x4, y4 = x, y + h
        
        # Add confidence score (2.0 as default)
        pred_str += f'{x1:.6f} {y1:.6f} 2.0 {x2:.6f} {y2:.6f} 2.0 {x3:.6f} {y3:.6f} 2.0 {x4:.6f} {y4:.6f} 2.0 '
    
    prediction_strings.append(pred_str.strip())

# Statistics
num_detections = sum(1 for ps in prediction_strings if ps)
total_gates = sum(len(pred) for pred in predictions)

print(f"\nPrediction Statistics:")
print(f"  Images with detections: {num_detections}/{len(predictions)}")
print(f"  Total gates detected: {total_gates}")
print(f"  Average gates per image: {total_gates / len(predictions):.2f}")

In [ ]:
# Create submission DataFrame
submission_df = pd.DataFrame({
    'Id': range(len(prediction_strings)),
    'PredictionString': prediction_strings
})

# Save submission file
submission_df.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("✓ Submission file 'submission.csv' created successfully!")
print("="*60)

# Show sample predictions
print("\nSample predictions (first 10):")
for i in range(min(10, len(prediction_strings))):
    num_boxes = len(predictions[i])
    print(f"  Image {i}: {num_boxes} gate(s) detected")

# Display first few rows
print("\nSubmission preview:")
print(submission_df.head())

## 13. Memory Cleanup

In [ ]:
# Clean up GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
gc.collect()

print("Memory cleanup complete!")